In [0]:
from pyspark.sql import functions as F

 ========== KONFİGÜRASYON ==========
catalog = "workspace"
source_schema = "default"
source_table = "silver_orders_clean"
target_schema = "gold"
target_table = "e_commerce"

key_cols = ["order_id"]
surrogate_key = "OrderKey"
cdc_col = "modifiedDate"

full_source = f"{source_schema}.{source_table}"
full_target = f"{catalog}.{target_schema}.{target_table}"

target_exists = spark.catalog.tableExists(full_target)

if target_exists:
    last_load_df = spark.sql(f"SELECT max({cdc_col}) as last_load FROM {full_target}")
    last_load_row = last_load_df.collect()[0]
    last_load = last_load_row["last_load"] if last_load_row["last_load"] else None
    
    if last_load:
        print(f"Son yüklenen tarih: {last_load}")
        df_src = spark.table(full_source).filter(F.col(cdc_col) > last_load)
    else:
        df_src = spark.table(full_source)
else:
    print("Hedef tablo yok, tüm kaynak verisi yükleniyor")
    df_src = spark.table(full_source)

row_count = df_src.count()
print(f"Kaynaktan okunan satır sayısı: {row_count}")

if row_count == 0:
    print("İşlenecek yeni veri yok.")
    dbutils.notebook.exit("No data")

 Mevcut max key değerini bul
if target_exists:
    max_key_df = spark.sql(f"SELECT max({surrogate_key}) as max_key FROM {full_target}")
    max_key_row = max_key_df.collect()[0]
    max_key = max_key_row["max_key"] if max_key_row["max_key"] else 0
else:
    max_key = 0

print(f"Max key: {max_key}")

 Sıra numarası oluştur
df_src = df_src.withColumn("row_num", F.monotonically_increasing_id())
df_src = df_src.withColumn(surrogate_key, (F.col("row_num") + max_key + 1).cast("long"))
df_src = df_src.drop("row_num")

 Tarih kolonlarını ekle
df_src = df_src.withColumn("create_date", F.current_timestamp())
df_src = df_src.withColumn("update_date", F.current_timestamp())

if target_exists:
     Mevcut tabloyu oku
    df_target = spark.table(full_target)
    
     Union yapmadan önce kolon tiplerini kontrol et
    for col_name in df_target.columns:
        if col_name in df_src.columns:
            target_type = df_target.schema[col_name].dataType
            df_src = df_src.withColumn(col_name, F.col(col_name).cast(target_type))
    
    df_final = df_target.unionByName(df_src, allowMissingColumns=True)
    
     Overwrite ile yaz
    df_final.write \
        .mode("overwrite") \
        .option("overwriteSchema", "false") \
        .saveAsTable(full_target)
else:
     İlk oluşturma
    df_src.write \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(full_target)

print(f"Veri başarıyla {full_target} tablosuna kaydedildi.")

 Kontrol
final_count = spark.table(full_target).count()
print(f"Hedef tablodaki toplam satır: {final_count}")

In [0]:
%sql
use catalog `workspace`; select * from `gold`.`e_commerce` where order_id = 'ORD-30002' ;

order_id,Order_Date,product_category,segment,Quantity,Sales,returned?,delivered_on_time?,region,country,shipping_type,us_state,order_channel,payment_method,profit,customer_id,Returned,Delivered_on_time,modifiedDate,__START_AT,__END_AT,update_date,OrderKey,create_date
ORD-30002,2024-06-26,Phone,Corporate,6,5962.11,N,Y,EMEA,France,Express,null,Partner,Credit Card,906.22,CUST-1146,N,Y,2026-03-11T15:25:33.617Z,2026-03-11T15:25:33.617Z,null,2026-03-11T16:15:38.647Z,8002,2026-03-11T16:15:38.647Z
